In [9]:
import tensorflow as tf
import os
import numpy as np
from constant import MODEL_DIR, COMPLETE_NAME

# Makes sure that for every run, we are getting the same results.
tf.random.set_seed(1234)

In [10]:
def save_model_safely(model, model_name, model_dir=MODEL_DIR):
    """Function that simply makes sure I do not overwrite a model.
    """
    print("Model Being Saved In Dir: ",MODEL_DIR)
    
    # checking to see if the directory path exist
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)
        
    # checking to make sure
    existing_models = os.listdir(model_dir)
    if model_name in existing_models:
        choice = input(f"'{model_name}' already exists in '{model_dir}'. Do you want to overwrite this model? (y/n): ").strip().lower()
        if choice != "y":
            print("Save cancelled!")
            return

    # see <https://www.tensorflow.org/guide/keras/serialization_and_saving>
    filepath = os.path.join(model_dir, model_name)
    model.save(filepath)
    print(f"Model Saved to {filepath}!")

In [11]:
# Checking what this file contains 
with np.load("notMNIST.npz", allow_pickle=True) as f:
    print("file:", f.files)

file: ['x_train', 'y_train', 'x_test', 'y_test']


In [12]:
print("--Get data--")
with np.load("notMNIST.npz", allow_pickle=True) as f:
    x_train, y_train = f['x_train'], f['y_train']
    x_test, y_test = f['x_test'], f['y_test']


--Get data--


In [13]:
print("--Process data--")
x_train, x_test = x_train / 255.0, x_test / 255.0

--Process data--


In [14]:
# I believe what we are doing is including a channel for
# tensorflow to showcase we are in grayscale, which is simply 
# the value of the pixel itself
print("Shape Before:", x_train.shape)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("Shape After:", x_train.shape)
input_shape = (28, 28, 1)

Shape Before: (60000, 28, 28)
Shape After: (60000, 28, 28, 1)


In [43]:
print("--Make model--")
# model = tf.keras.models.Sequential([
#   tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
#   tf.keras.layers.Dense(128,activation='relu'), # see<https://www.tensorflow.org/datasets/keras_example>
#   tf.keras.layers.Dense(64,activation='relu'), # see<https://www.tensorflow.org/datasets/keras_example>
#   tf.keras.layers.Dense(32,activation='relu'), # see<https://www.tensorflow.org/datasets/keras_example>
#   tf.keras.layers.Dropout(0.2),
#   tf.keras.layers.Dense(10, activation='softmax')
# ])
# A D J
# thank you victor haou
# see<https://victorzhou.com/blog/keras-cnn-tutorial/#:~:text=%5D)%20The%20Sequential%20constructor%20takes%20an%20array,3%20types%20of%20layers.%20from%20tensorflow.%20keras.>
num_filters = 8
filter_size = 3
pool_size = 2
model = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(num_filters, filter_size, input_shape=(28, 28, 1)),
  tf.keras.layers.Conv2D(num_filters, filter_size),
  tf.keras.layers.MaxPooling2D(pool_size=pool_size),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(10, activation='softmax')
])
# model = tf.keras.models.Sequential([
#   tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
#   tf.keras.layers.Dense(128, activation='relu'),
#   tf.keras.layers.Dropout(0.2),
#   tf.keras.layers.Dense(10, activation='softmax')
# ])
#model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("--Fit model--")
model.fit(x_train, y_train, epochs=20, verbose=2)

--Make model--
--Fit model--
Epoch 1/20
1875/1875 - 8s - 4ms/step - accuracy: 0.8365 - loss: 0.5624
Epoch 2/20
1875/1875 - 7s - 4ms/step - accuracy: 0.8797 - loss: 0.4079
Epoch 3/20
1875/1875 - 6s - 3ms/step - accuracy: 0.8941 - loss: 0.3536
Epoch 4/20
1875/1875 - 6s - 3ms/step - accuracy: 0.9036 - loss: 0.3137
Epoch 5/20
1875/1875 - 7s - 4ms/step - accuracy: 0.9133 - loss: 0.2825
Epoch 6/20
1875/1875 - 6s - 3ms/step - accuracy: 0.9202 - loss: 0.2573
Epoch 7/20
1875/1875 - 7s - 3ms/step - accuracy: 0.9262 - loss: 0.2347
Epoch 8/20
1875/1875 - 7s - 4ms/step - accuracy: 0.9314 - loss: 0.2176
Epoch 9/20
1875/1875 - 7s - 4ms/step - accuracy: 0.9354 - loss: 0.2004
Epoch 10/20
1875/1875 - 7s - 4ms/step - accuracy: 0.9401 - loss: 0.1872
Epoch 11/20
1875/1875 - 5s - 3ms/step - accuracy: 0.9424 - loss: 0.1771
Epoch 12/20
1875/1875 - 6s - 3ms/step - accuracy: 0.9462 - loss: 0.1661
Epoch 13/20
1875/1875 - 6s - 3ms/step - accuracy: 0.9493 - loss: 0.1566
Epoch 14/20
1875/1875 - 7s - 4ms/step - accu

In [41]:
print("--Evaluate model--")
model_loss1, model_acc1 = model.evaluate(x_train,  y_train, verbose=2)
model_loss2, model_acc2 = model.evaluate(x_test,  y_test, verbose=2)
print(f"Train / Test Accuracy: {model_acc1*100:.1f}% / {model_acc2*100:.1f}%")

--Evaluate model--
1875/1875 - 4s - 2ms/step - accuracy: 0.9598 - loss: 0.1368
313/313 - 1s - 2ms/step - accuracy: 0.9408 - loss: 0.2243
Train / Test Accuracy: 96.0% / 94.1%


In [42]:
save_model_safely(model,COMPLETE_NAME)

Model Being Saved In Dir:  model


'notMNIST-Complete.keras' already exists in 'model'. Do you want to overwrite this model? (y/n):  y


Model Saved to model/notMNIST-Complete.keras!
